## مقارنة أداء البحث: FAISS مقابل البحث اليدوي

هذا الـ Notebook مخصص لتحقيق أحد المتطلبات الإضافية: إثبات فعالية استخدام **Vector Store (FAISS)** مقارنة بالبحث اليدوي (Brute-force) عن التشابه.

**الهدف:** مقارنة الطريقتين من حيث:
1.  **السرعة (Speed):** قياس زمن الاستجابة لكل عملية بحث.
2.  **الدقة (Accuracy):** التأكد من أن FAISS يعطي نفس دقة البحث اليدوي (أو دقة قريبة جداً).

### الخطوة 1: الإعداد وتحميل النماذج والبيانات

In [1]:
import requests
import ir_datasets
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import os
import joblib
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- الإعدادات ---
DATASET_NAME = 'lotte/lifestyle/dev' # تأكد من أن النماذج لهذه المجموعة قد تم بناؤها
IR_DATASET_NAME_TEST = 'lotte/lifestyle/dev'
QUERY_SAMPLE_SIZE = 50 # نستخدم عينة صغيرة لقياس السرعة والتحقق من الدقة

# --- الحل: تحديد المسار الرئيسي للمشروع بشكل صحيح ---
# هذا يضمن أننا نستطيع الوصول إلى أي مجلد من داخل الـ Notebook
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(project_root)

from config import API_PORTS, MODELS_DIR, EMBEDDING_MODEL_NAME

SEARCH_API_URL = f"http://127.0.0.1:{API_PORTS['SEARCH']}/search/"

# --- الحل: بناء المسار المطلق لمجلد النماذج ---
MODEL_DIR = os.path.join(project_root, MODELS_DIR, DATASET_NAME.replace('/', '_'))


# --- تحميل النماذج محلياً لإجراء البحث اليدوي ---
print(f"Loading models from: {MODEL_DIR}")
bert_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
doc_ids = joblib.load(os.path.join(MODEL_DIR, 'doc_ids.joblib'))
full_embeddings_matrix = np.load(os.path.join(MODEL_DIR, 'bert_embeddings.npy'))
print("Models loaded successfully.")

# --- تحميل بيانات الاختبار ---
dataset = ir_datasets.load(IR_DATASET_NAME_TEST)
queries = {q.query_id: q.text for q in dataset.queries_iter()}
query_ids_to_run = list(queries.keys())[:QUERY_SAMPLE_SIZE]
print(f"Test data loaded. Running comparison on {len(query_ids_to_run)} queries.")


Loading models from: /home/hussam/information-retrieval-system/models_v2/lotte_lifestyle_dev
Models loaded successfully.


AttributeError: queries_iter

### الخطوة 2: تعريف دوال البحث (FAISS واليدوي)

In [2]:
def search_with_faiss(query_text):
    """Calls the API which uses the fast FAISS index."""
    payload = {"dataset_name": DATASET_NAME, "query": query_text, "model_type": "bert", "top_k": 10}
    start_time = time.time()
    response = requests.post(SEARCH_API_URL, json=payload)
    end_time = time.time()
    duration = end_time - start_time
    results = [res['doc_id'] for res in response.json().get('results', [])]
    return results, duration

def search_manually(query_text):
    """Performs brute-force similarity search locally."""
    start_time = time.time()
    
    # 1. Encode query
    query_embedding = bert_model.encode([query_text])
    
    # 2. Calculate cosine similarity against ALL document embeddings
    similarities = cosine_similarity(query_embedding, full_embeddings_matrix).flatten()
    
    # 3. Get top 10 results
    top_indices = np.argsort(similarities)[-10:][::-1]
    results = [doc_ids[i] for i in top_indices]
    
    end_time = time.time()
    duration = end_time - start_time
    return results, duration

### الخطوة 3: تنفيذ المقارنة

In [3]:
comparison_results = []

for query_id in tqdm(query_ids_to_run, desc="Comparing Search Methods"):
    query_text = queries[query_id]
    
    faiss_results, faiss_time = search_with_faiss(query_text)
    manual_results, manual_time = search_manually(query_text)
    
    # حساب الدقة (هل النتائج متطابقة؟)
    # قد يكون هناك اختلافات طفيفة جداً بسبب دقة الحسابات، لذا نقارن أول 5 نتائج مثلاً
    precision = len(set(faiss_results[:5]).intersection(set(manual_results[:5]))) / 5.0
    
    comparison_results.append({
        'Query': query_text[:40] + '...',
        'FAISS Time (s)': faiss_time,
        'Manual Time (s)': manual_time,
        'Speedup Factor': manual_time / faiss_time if faiss_time > 0 else float('inf'),
        'Precision Match @5': precision
    })

comparison_df = pd.DataFrame(comparison_results)

# حساب المتوسطات
avg_faiss_time = comparison_df['FAISS Time (s)'].mean()
avg_manual_time = comparison_df['Manual Time (s)'].mean()
avg_speedup = comparison_df['Speedup Factor'].mean()
avg_precision = comparison_df['Precision Match @5'].mean()

Comparing Search Methods: 100%|██████████████████████████████████████████████████████████████████████████████| 50/50 [00:27<00:00,  1.79it/s]


### الخطوة 4: عرض النتائج النهائية

In [4]:
print("--- FAISS vs. Manual Search Performance Comparison ---")
print("\nAverage Time per Query:")
print(f"  - FAISS (API Call): {avg_faiss_time:.4f} seconds")
print(f"  - Manual (Brute-force): {avg_manual_time:.4f} seconds")
print("\nPerformance Metrics:")
print(f"  - Average Speedup Factor: {avg_speedup:.2f}x (FAISS is faster)")
print(f"  - Average Precision Match @5: {avg_precision:.2%} (How similar are the results)")

print("\nDetailed Results per Query:")
display(comparison_df.style.format({
    'FAISS Time (s)': '{:.4f}',
    'Manual Time (s)': '{:.4f}',
    'Speedup Factor': '{:.2f}x',
    'Precision Match @5': '{:.0%}'
}))

--- FAISS vs. Manual Search Performance Comparison ---

Average Time per Query:
  - FAISS (API Call): 0.0672 seconds
  - Manual (Brute-force): 0.4910 seconds

Performance Metrics:
  - Average Speedup Factor: 7.23x (FAISS is faster)
  - Average Precision Match @5: 99.20% (How similar are the results)

Detailed Results per Query:


,Query,FAISS Time (s),Manual Time (s),Speedup Factor,Precision Match @5
0,how can we get concentration onsomething...,0.1318,1.7081,12.96x,100%
1,Why doesn't the water fall off earth if...,0.1066,0.8790,8.25x,100%
2,How do I determine the charge of the iro...,0.0709,0.5249,7.40x,100%
3,I have mice.How do I get rid of them hum...,0.0687,0.5276,7.68x,100%
4,"What does ""see Leaflet"" mean on Ept Preg...",0.0664,0.4803,7.23x,100%
5,what is innate immunity?...,0.0604,0.4659,7.71x,100%
6,how can i lose 30 pounds by june3?...,0.0620,0.4792,7.72x,80%
7,What are the words to write the sound of...,0.0672,0.4867,7.25x,100%
8,Why must I have an uncracked winshield i...,0.0670,0.4479,6.68x,100%
9,What is Blaphsemy?...,0.0581,0.4614,7.94x,100%
